In [1]:
from NoisyCircuits.utils.CreateNoiseModel import CreateNoiseModel, GetNoiseModel
from NoisyCircuits.utils.GetNoiseModel import GetNoiseModel as GNM
import pandas as pd
import numpy as np
import os
import json

2026-03-04 16:41:15,141	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
api_json = json.load(open(os.path.join(os.path.expanduser("~"), "ibm_api.json"), "r"))
token = api_json["apikey"]
service_crn = api_json["service-crn"]

In [3]:
noise_model_api = GetNoiseModel(backend_name="ibm_marrakesh", token=token, service_crn=service_crn).get_noise_model(save_csv=True, file_name="Noise_Model_from_API.csv")

In [4]:
noise_model_qiskit = GNM(backend_name="ibm_marrakesh", token=token).get_noise_model()

qiskit_runtime_service._discover_account:WARNING:2026-03-04 16:41:33,226: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-03-04 16:41:39,740: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Open_Sys. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-04 16:41:39,742: Using instance: Open_Sys, plan: open
qiskit_runtime_service.backends:WARNING:2026-03-04 16:41:39,968: Using instance: Open_Sys, plan: open


In [ ]:
def compare_noise(model1, model2):
    instructions1 = model1["instructions"]
    instructions2 = model2["instructions"]
    probs1 = model1["probabilities"]
    probs2 = model2["probabilities"]
    for instr1, prob1 in zip(instructions1, probs1):
        pass

In [12]:
from qiskit_aer.noise import thermal_relaxation_error, depolarizing_error, NoiseModel
from qiskit.quantum_info import average_gate_fidelity

t1 = 151.67
t2 = 89.16

thermal_error = thermal_relaxation_error(t1*1e-6, t2*1e-6, 36e-9)
f_thermal = average_gate_fidelity(thermal_error)
dim = 2
gate_error = 1.616e-4
p = dim * (gate_error - (1 - f_thermal)) / (dim * f_thermal - 1)
p = min(4/3, max(0, p))
print("Depolarizing error probability:", p)
depol_error = depolarizing_error(p, 1)
m = NoiseModel()
m.add_quantum_error(depol_error.compose(thermal_error), "x", [0])
# m.add_quantum_error(thermal_error, "x", [0])
mdict = m.to_dict(serializable=True)

Depolarizing error probability: 0


In [13]:
mdict["errors"][0]

{'type': 'qerror',
 'id': 'e0ca67855b1146249dcad9ed32bbe3dc',
 'operations': ['x'],
 'instructions': [[{'name': 'id', 'qubits': [0]},
   {'name': 'id', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}]],
 'probabilities': [0.9996794918719024,
  8.317887442555663e-05,
  0.00023732925367203617],
 'gate_qubits': [[0]]}

In [27]:
noise_model_qiskit["errors"][82+156]

{'type': 'qerror',
 'id': 'cb548ea681ee48e8b67354ab8d956204',
 'operations': ['x'],
 'instructions': [[{'name': 'x', 'qubits': [0]},
   {'name': 'id', 'qubits': [0]}],
  [{'name': 'x', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'x', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}],
  [{'name': 'y', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'y', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'y', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}]],
 'probabilities': [0.33149046724220904,
  1.5070231114920536e-05,
  0.0018277958600093997,
  0.33149046724220904,
  1.5070231114920536e-05,
  0.0018277958600093997,
  0.33149046724220904,
  1.5070231114920536e-05,
  0.0018277958600093997],
 'gate_qubits': [[82]]}

In [13]:
data_api = pd.read_csv("Noise_Model_from_API.csv")
data_downloaded = pd.read_csv("calibration_data.csv")

In [ ]:
for col in data_api.columns:
    col_api = data_api[col].values
    col_downloaded = data_downloaded[col].values
    try:
        if np.allclose(col_api, col_downloaded, atol=1e-10):
            print(f"Column '{col}' is approximately equal in both dataframes.")
        else:
            print(f"Column '{col}' is NOT approximately equal in both dataframes.")
            for i in range(len(col_api)):
                if not np.isclose(col_api[i], col_downloaded[i], atol=1e-10):
                    print(f"{col} has different values at index {i}: API value = {col_api[i]}, Downloaded value = {col_downloaded[i]}")
    except Exception as e:
        print(f"Error comparing column '{col}': {e}")

Column 'Qubit' is approximately equal in both dataframes.
Column 'T1 (us)' is approximately equal in both dataframes.
Column 'T2 (us)' is approximately equal in both dataframes.
Column 'Prob meas 0 prep 1' is approximately equal in both dataframes.
Column 'Prob meas 1 prep 0' is approximately equal in both dataframes.
Column 'Single Qubit Gate Length (ns)' is approximately equal in both dataframes.
Column 'x Error' is NOT approximately equal in both dataframes.
x Error has different values at index 82: API value = 1.0, Downloaded value = nan
x Error has different values at index 113: API value = 1.0, Downloaded value = nan
x Error has different values at index 119: API value = 1.0, Downloaded value = nan
x Error has different values at index 130: API value = 1.0, Downloaded value = nan
Column 'sx Error' is NOT approximately equal in both dataframes.
sx Error has different values at index 82: API value = 1.0, Downloaded value = nan
sx Error has different values at index 113: API value =

In [15]:
col_names_two_qubit = [
    "Gate Length (ns)",
    "cz Error", 
    "rzz Error"
]

for col in col_names_two_qubit:
    col_api = data_api[col].values
    col_downloaded = data_downloaded[col].values
    for i in range(len(col_api)):
        if col_api[i] != col_downloaded[i]:
            print(f"{col} has different values at index {i}: API value = {col_api[i]}, Downloaded value = {col_downloaded[i]}")

cz Error has different values at index 82: API value = 81:0.09323549120560648;83:1, Downloaded value = 81:0.09323549120560648
cz Error has different values at index 83: API value = 96:0.006754141154095061;82:1;84:0.0028796826155672306, Downloaded value = 96:0.006754141154095061;84:0.0028796826155672306
cz Error has different values at index 113: API value = 119:1;112:0.0015704252416779418;114:1, Downloaded value = 112:0.0015704252416779418
cz Error has different values at index 114: API value = 115:0.009251915676606659;113:1, Downloaded value = 115:0.009251915676606659
cz Error has different values at index 119: API value = 113:1;133:0.07539295870284293, Downloaded value = 133:0.07539295870284293
cz Error has different values at index 129: API value = 130:1;128:0.0024604146706956598;118:0.0021525569921425347, Downloaded value = 128:0.0024604146706956598;118:0.0021525569921425347
cz Error has different values at index 130: API value = 129:1;131:1, Downloaded value = nan
cz Error has dif

In [16]:
noise_model_api["errors"][0]

{'type': 'qerror',
 'id': '9f4ddf66911546f886148acfa762b1cd',
 'operations': ['x'],
 'instructions': [[{'name': 'id', 'qubits': [0]},
   {'name': 'id', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'x', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'y', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'x', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'y', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'reset', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'reset', 'qubits': [0]}, {'name': 'x', 'qubits': [0]}],
  [{'name': 'reset', 'qubits': [0]}, {'name': 'y', 'qubits': [0]}],
  [{'name': 'reset', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}]],
 'probabilities': [0.9992826779679655,
  0.00013092688909617967,
  0.00013092688909617967,
  0.0001309268

In [17]:
noise_model_qiskit["errors"][0]

{'type': 'qerror',
 'id': 'd5fc42231a13408eb5594aa34cc46907',
 'operations': ['id'],
 'instructions': [[{'name': 'id', 'qubits': [0]},
   {'name': 'id', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'id', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}],
  [{'name': 'x', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'x', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'x', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}],
  [{'name': 'y', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'y', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'y', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'id', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'z', 'qubits': [0]}],
  [{'name': 'z', 'qubits': [0]}, {'name': 'reset', 'qubits': [0]}]],
 'probabilities': [0.9991891346141385,
  0.00022438168285514437,
  0.00010000179838950698,
  0.000162108